# Scraping Pilled.net

---

[Pilled.net](https://pilled.net) is a small social media platform popular among conspiracy theorists. Unlike Twitter/X or Facebook, it has only ~1 million posts and minimal anti-scraping measures. That makes it a rare thing: **a social media platform you can realistically scrape in its entirety.**

In this workshop, we'll:

1. Learn to see that everything on a web page is just **rendered by calls to the network**
2. Reverse-engineer pilled.net's API endpoints using browser DevTools
3. Build a scraper in Python to collect posts
4. Do quick analysis on what we find

### The scenario

Your editor says: *"A far-right group has been in the news. What have they been saying online?"*

---

## Setup

### Software you need

- **Chrome** (for DevTools)
- **VS Code** with Jupyter notebook support
- **uv** (Python package manager)

Run in your terminal:

```bash

# Set up the project
uv init pilled-scraper
cd pilled-scraper
uv venv
source .venv/bin/activate

# Install dependencies
uv add requests pandas jupyter ipykernel

# Optional but useful:
uv add yt-dlp   # for downloading media if we have time

# Register the Jupyter kernel
uv run python -m ipykernel install --user --name pilled-scraper
```

In [ ]:
import requests
import pandas as pd
import json
import time
from datetime import datetime
from pathlib import Path

print("Ready to go.")

---
## Part 1: Everything on the page is a network call

When you load a page on pilled.net (or almost any modern web app), the HTML that comes back is basically empty — just JavaScript bootstrap code. The actual content (posts, profiles, comments, images) all gets loaded by **background API calls** that return JSON.

That means: if you can see the call, you can replay it. And if you can replay it, you can scrape it.

### Step 1: Go to the site

Open https://pilled.net in Chrome. Navigate to the Patriot Party Pod's posts:

https://pilled.net/patriotpartypod/posts


### Step 2: Open DevTools

- `F12` or `Cmd+Option+I` (Mac) / `Ctrl+Shift+I` (Windows)
- Click the **Network** tab
- Click the **Fetch/XHR** filter (this hides images, CSS, etc. and shows only data calls)

### Step 3: Reload the page

Hit `Cmd+R`. Watch the requests populate in the Network panel.

**Every line that appears is the browser asking the server for data.** The page you see is just those responses stitched together by JavaScript.

### Step 4: Find the post data

Click through the requests. For each one, check the **Response** tab. You're looking for JSON that contains the post text you can see on screen.

**Pro tip: Use `Cmd+F` in the Network panel.** You can search through response bodies. Copy a distinctive phrase from a post on the page, search for it, and you'll find exactly which network request delivered that content.



### Exercise: Copy as cURL

1. Right-click the posts request → **Copy** → **Copy as cURL**
2. Paste into terminal, hit Enter
3. Does it return the same JSON?

Paste what you found below:

In [ ]:
# Paste your cURL here as a comment:
#
# curl 'https://pilled-lqs-api.pilled.net/topic/getUserTopics/169214/0?filter=recent&search=undefined' \
#   -H 'sec-ch-ua-platform: "macOS"' \
#   -H 'Referer: https://pilled.net/' \
#   -H 'User-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36' \
#   -H 'Accept: application/json, text/plain, */*' \
#   -H 'sec-ch-ua: "Not:A-Brand";v="99", "Google Chrome";v="145", "Chromium";v="145"' \
#   -H 'DNT: 1' \
#   -H 'sec-ch-ua-mobile: ?0'
# #
# What endpoint did you find?
# What parameters does it take?


---
## Part 2: From browser to notebook

Now we move from DevTools to Python. The pattern:

1. Translate the cURL into `requests.get()` or `requests.post()`
2. Verify you get the same data
3. Build the loop

### Configuration

Fill in from your DevTools findings:

In [ ]:
# ============================================================
# CONFIGURATION — fill in from DevTools
# ============================================================

BASE_URL = "https://pilled-lqs-api.pilled.net"

# The endpoint you found
POSTS_ENDPOINT = "/topic/getUserTopics/169214/0?filter=recent&search=undefined" 

# Headers — copy the important ones from DevTools
HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Origin": "https://pilled.net",
    "Referer": "https://pilled.net/",
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),

}

PAGE_SIZE = 20  # match what you saw in DevTools
DELAY = 2       # seconds between requests — be nice! 


### Fetch one page and inspect

In [ ]:
def test_request():
    """Replay one API call and inspect what comes back."""
    url = f"{BASE_URL}{POSTS_ENDPOINT}"
    params = {
        "page": 1,
        "pageSize": PAGE_SIZE,
    }

    print(f"GET {url}")
    print(f"Params: {params}\n")

    resp = requests.get(url, params=params, headers=HEADERS, timeout=30)
    print(f"Status: {resp.status_code}")
    print(f"Content-Type: {resp.headers.get('content-type')}")

    # Check for rate limit headers
    for h in resp.headers:
        if 'rate' in h.lower() or 'limit' in h.lower() or 'retry' in h.lower():
            print(f"⚠ Rate limit header → {h}: {resp.headers[h]}")

    if resp.status_code == 200:
        data = resp.json()
        if isinstance(data, dict):
            print(f"\nTop-level keys: {list(data.keys())}")
            for key, val in data.items():
                if isinstance(val, list):
                    print(f"  '{key}' → list of {len(val)} items")
                    if val and isinstance(val[0], dict):
                        print(f"    First item keys: {list(val[0].keys())}")
                else:
                    print(f"  '{key}' → {type(val).__name__}: {str(val)[:80]}")
        elif isinstance(data, list):
            print(f"\nResponse is a list of {len(data)} items")
            if data and isinstance(data[0], dict):
                print(f"First item keys: {list(data[0].keys())}")
        return data
    else:
        print(f"\nError: {resp.text[:500]}")
        return None

sample = test_request()

In [ ]:
def look_at_one_post(data):
    """Pretty-print one post so we can see field names."""
    posts = data if isinstance(data, list) else data.get("posts", data.get("items", [data]))
    if not posts:
        print("No posts found.")
        return
    print(json.dumps(posts[0], indent=2, default=str)[:3000])

look_at_one_post(sample)

---
## Part 3: Paginate and collect everything

In [ ]:
def scrape_user_posts(max_pages):
    """Scrape all posts from a user. Returns list of raw dicts."""
    all_posts = []
    url = f"{BASE_URL}{POSTS_ENDPOINT}"
    session = requests.Session()
    session.headers.update(HEADERS)

    for page in range(1, max_pages + 1):
        params = {
            "page": page,
            "pageSize": PAGE_SIZE,
        }
        print(f"  Page {page}...", end=" ", flush=True)

        try:
            resp = session.get(url, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
        except requests.RequestException as e:
            print(f"error: {e}")
            time.sleep(5)
            try:
                resp = session.get(url, params=params, timeout=30)
                resp.raise_for_status()
                data = resp.json()
            except Exception:
                print("retry failed, stopping.")
                break

        posts = data if isinstance(data, list) else data.get("posts", data.get("items", []))
        if not posts:
            print("done.")
            break

        all_posts.extend(posts)
        print(f"{len(posts)} posts (total: {len(all_posts)})")

        if len(posts) < PAGE_SIZE:
            print("  Last page.")
            break

        time.sleep(DELAY)

    print(f"\nCollected {len(all_posts)} posts.")
    return all_posts

raw_posts = scrape_user_posts(max_pages=2)

In [ ]:
# Save raw JSON 
def save_raw(posts, filename="raw_posts.json"):
    with open(filename, "w") as f:
        json.dump(posts, f, indent=2, default=str)
    print(f"Saved {len(posts)} posts → {filename}")

save_raw(raw_posts)

### Backup plan: load from a local file

If the live scrape isn't working (auth problems, site is down, rate limits), load a pre-scraped file:

In [ ]:
# with open("raw_posts.json") as f:
#     raw_posts = json.load(f)
# print(f"Loaded {len(raw_posts)} posts from file")

---
## Part 4: Clean the data



In [ ]:
def clean_posts(raw_posts):
    """Raw JSON → clean DataFrame. Field names matched to pilled.net API."""
    records = []
    for post in raw_posts:
        # Extract text from proofs (TextProof entries)
        text = ""
        has_image = False
        has_video = False
        proofs = post.get("proofs", [])
        for proof in proofs:
            if proof.get("ProofType") == "TextProof" and proof.get("Text"):
                text = proof["Text"]
            if proof.get("BlobFileCategory") == "video" or proof.get("VideoUri"):
                has_video = True
            if proof.get("BlobFileCategory") == "image" or proof.get("ImageUri"):
                has_image = True

        # Pull engagement from topicFacts (more reliable than top-level fields)
        facts = post.get("topicFacts", {})

        record = {
            "post_id":          post.get("topicID"),
            "user_id":          post.get("userID"),
            "username":         post.get("userName"),
            "text":             text,
            "title":            post.get("topicName", ""),
            "created_at":       post.get("createDateTime"),
            "likes":            facts.get("netPills", post.get("netPills", 0)),
            "comments":         facts.get("numOfComments", post.get("numOfComments", 0)),
            "shares":           facts.get("reposts", post.get("reposts", 0)),
            "quotes":           facts.get("quotes", post.get("quotes", 0)),
            "impressions":      facts.get("impressions", post.get("views", 0)),
            "views":            facts.get("views", post.get("views", 0)),
            "live_views":       facts.get("liveViews", post.get("liveViews", 0)),
            "gold_pills":       facts.get("goldPills", post.get("goldPillAmount", 0)),
            "red_pills":        facts.get("redpillCount", post.get("redPilledCount", 0)),
            "has_image":        has_image,
            "has_video":        has_video,
            "has_link":         False,  # not a distinct field; embedded in TextProof text if present
            "is_stream":        post.get("isStream", False),
            "is_live_now":      post.get("isLiveNow", False),
            "is_shared":        bool(post.get("quoteOnTopicID") or post.get("commentOnTopicID")),
            "quote_of_id":      post.get("quoteOnTopicID"),
            "comment_of_id":    post.get("commentOnTopicID"),
            "is_nsfw":          post.get("nsfwFlagged", False),
            "is_verified":      post.get("isVerified", False),
            "primary_category": post.get("primaryCategory", {}).get("name"),
            "topic_guid":       post.get("topicGuid"),
        }
        records.append(record)

    df = pd.DataFrame(records)

    if df["created_at"].notna().any():
        df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce", utc=True)

    df = df.sort_values("created_at", ascending=False).reset_index(drop=True)
    return df


df = clean_posts(raw_posts)
df.head()

### Export to TSV → Google Sheets

TSV imports cleaner than CSV when posts contain commas and quotes.

In [ ]:
def export_tsv(df, filename="posts.tsv"):
    cols = [
        "post_id", "username", "created_at", "title", "text",
        "likes", "red_pills", "gold_pills", "comments", "shares", "quotes",
        "views", "live_views", "impressions",
        "has_image", "has_video",
        "is_stream", "is_live_now", "is_shared", "is_nsfw", "is_verified",
        "quote_of_id", "comment_of_id",
        "primary_category", "topic_guid",
    ]
    cols = [c for c in cols if c in df.columns]
    df[cols].to_csv(filename, sep="\t", index=False)
    print(f"Exported {len(df)} rows → {filename}")
    print("Open in Google Sheets: File > Import > Upload")

export_tsv(df)

---
## Part 5: Analysis

Editor asked: *"What have they been saying?"*

### When are they posting?

In [ ]:
def posting_activity(df):
    print(f"Total posts: {len(df)}")
    print(f"Streams:     {df['is_stream'].sum()}")
    print(f"Verified:    {df['is_verified'].sum()}")

    if df["created_at"].notna().any():
        print(f"First post: {df['created_at'].min()}")
        print(f"Last post:  {df['created_at'].max()}")

        print("\n--- Posts per month ---")
        monthly = df.set_index("created_at").resample("ME").size()
        for date, count in monthly.items():
            bar = "█" * min(count, 60)
            print(f"  {date.strftime('%Y-%m')}: {count:3d} {bar}")

        print("\n--- By hour of day (EST) ---")
        est = df["created_at"].dt.tz_convert("America/New_York")
        for hour, count in est.dt.hour.value_counts().sort_index().items():
            bar = "█" * min(count, 40)
            print(f"  {hour:02d}:00  {count:3d} {bar}")

        print("\n--- By day of week ---")
        days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
        for d, count in df["created_at"].dt.dayofweek.value_counts().sort_index().items():
            bar = "█" * min(count, 40)
            print(f"  {days[d]}  {count:3d} {bar}")

        print("\n--- Engagement summary ---")
        for col in ["likes", "red_pills", "gold_pills", "comments", "shares", "quotes", "views", "impressions"]:
            if col in df.columns:
                print(f"  {col:12s}  total={df[col].sum():6.0f}  mean={df[col].mean():5.1f}  max={df[col].max():5.0f}")

        if "primary_category" in df.columns:
            print("\n--- By category ---")
            for cat, count in df["primary_category"].value_counts().items():
                bar = "█" * min(count, 40)
                print(f"  {str(cat):20s}  {count:3d} {bar}")

posting_activity(df)

### Who are they sharing / boosting?

In [ ]:
def sharing_analysis(df):
    original = df[(df["is_shared"] == False) & df["quote_of_id"].isna() & df["comment_of_id"].isna()]
    quotes    = df[df["quote_of_id"].notna()]
    comments  = df[df["comment_of_id"].notna()]

    print(f"Original posts:  {len(original)}")
    print(f"Quote posts:     {len(quotes)}")
    print(f"Comment replies: {len(comments)}")

    if len(quotes) > 0:
        print(f"\nQuoted post IDs (top 15):")
        for pid, count in quotes["quote_of_id"].value_counts().head(15).items():
            bar = "█" * min(count, 30)
            print(f"  {str(pid):15s} {count:3d} {bar}")

    if len(comments) > 0:
        print(f"\nMost replied-to post IDs (top 15):")
        for pid, count in comments["comment_of_id"].value_counts().head(15).items():
            bar = "█" * min(count, 30)
            print(f"  {str(pid):15s} {count:3d} {bar}")

sharing_analysis(df)

### Impressions / engagement

In [ ]:
def engagement_summary(df):
    for col in ["likes", "red_pills", "gold_pills", "comments", "shares", "quotes", "views", "live_views", "impressions"]:
        if col in df.columns:
            vals = pd.to_numeric(df[col], errors='coerce').fillna(0)
            if vals.sum() > 0:
                print(f"  {col:12s}  total={vals.sum():7.0f}  mean={vals.mean():5.1f}  max={vals.max():.0f}")

    for metric in ["likes", "impressions", "views"]:
        if metric not in df.columns:
            continue
        print(f"\nTop 5 by {metric}:")
        for _, row in df.nlargest(5, metric).iterrows():
            title = (row.get("title") or "")[:80]
            text  = (row.get("text")  or "")[:80]
            ts    = row.get("created_at", "?")
            print(f"  {row[metric]:.0f} {metric} | {ts} | stream={row.get('is_stream', False)}")
            if title:
                print(f"    title: {title}")
            if text:
                print(f"    text:  {text}")
            print()

engagement_summary(df)

---
## Bonus: Downloading media

Useful for archiving before things get deleted.

In [ ]:
import requests
from pathlib import Path

def download_video(post_id, video_url, output_dir="media"):
    Path(output_dir).mkdir(exist_ok=True)
    output_path = Path(output_dir) / f"{post_id}.mp4"
    if output_path.exists():
        print(f"  SKIP (exists): {output_path}")
        return
    try:
        with requests.get(video_url, stream=True, timeout=30, headers=HEADERS) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length", 0))
            downloaded = 0
            with open(output_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    f.write(chunk)
                    downloaded += len(chunk)
            mb = downloaded / 1024 / 1024
            print(f"  OK: {output_path} ({mb:.1f} MB)")
    except Exception as e:
        print(f"  FAILED {post_id}: {e}")

raw_posts_by_id = {p["topicID"]: p for p in raw_posts}

video_urls = []
for _, row in df[df["has_video"]].iterrows():
    for proof in raw_posts_by_id[row["post_id"]]["proofs"]:
        if proof.get("VideoUri"):
            video_urls.append((row["post_id"], proof["VideoUri"]))

# print(f"Downloading {len(video_urls)} videos...")
# for post_id, url in video_urls:
#     print(f"  {post_id}: {url}")
#     download_video(post_id, url)
#     time.sleep(0.5)

---
## Cheat sheet

| What to look for | Where |
|---|---|
| API endpoint URL | Network tab → Request URL |
| Pagination | Query params: `page`, `offset`, `cursor` |
| Outgoing params/payload | Headers or Payload tab |
| Auth tokens/cookies | Request Headers → `Authorization`, `Cookie` |
| Rate limiting | Response Headers → `X-RateLimit-*`, `Retry-After` |
| Total count | Response body → `totalCount`, `hasMore` |

```
Browse site → DevTools Network → Find the call →
Copy as cURL → Test in terminal → Python requests →
Paginate → Save raw JSON → Clean DataFrame →
Export TSV → Google Sheets → Write the story
```